<a href="https://colab.research.google.com/github/Syamendrra/SeedVR2ImageUpscalerGoogleCollab/blob/main/SeedVR2_Image_Upscaler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 SeedVR2 Image Upscaler: User Guide & Documentation

Welcome to the **SeedVR2 Video/Image Upscaling** pipeline. This notebook leverages the **SeedVR2 7B GGUF** model to upscale, denoise, and sharpen images up to 4K resolution using your Google Colab GPU runtime.

---

## 📋 Step-by-Step Pipeline Overview

1. **Environment Check (Cell 1):** Verifies PyTorch, CUDA, and GPU hardware capabilities (e.g., Tesla T4 with ~15GB VRAM).
2. **Setup and Installation (Cell 2):** Clones the codebase repository, installs required dependencies, and configures the environment.
3. **CLI Verification (Cell 3):** Performs a validation run of the upscaling engine CLI (`inference_cli.py`) to verify argument compatibility.
4. **Upload Input Images (Cell 4):** Prompts you to upload a `.zip` archive containing your source images, extracts them securely, and verifies the image paths.
5. **Batch Processing Run (Cell 5):** Loops over every extracted image, running the SeedVR2 engine to upscale it, while saving the outputs.
6. **Result Verification (Cell 7):** Scans the output folder to show the resolution of the newly generated files.
7. **Packaging & Cleanup (Cell 6):** Archives your upscaled files into a download-ready ZIP file and automatically wipes the temporary server folders so your next run is clean and memory-efficient.

---

## ⚙️ Key Parameters Explained (How to Customize)

You can customize the upscaling behavior in the **Batch Processing Run** cell (Cell 5) by modifying the command-line arguments in the `cmd` list:

### 📏 Resolution & Quality
* **`--resolution "2160"`**
  * **What it does:** Sets the target shorter-dimension (or reference scale) for the upscaled image.
  * **How to change:** Change `"2160"` (which outputs 4K class images) to `"1080"` for 1080p outputs, or `"1440"` for 2K outputs.
* **`--max_resolution "3840"`**
  * **What it does:** Sets a safety ceiling on the longest dimension of the output image to prevent Out-Of-Memory (OOM) crashes.
  * **How to change:** Scale this up or down in proportion to your target aspect ratio (e.g., `"1920"` for 1080p, `"3840"` for 4K).

### 🚀 Speed & GPU Memory (VRAM) Management
Depending on your GPU, you can manage memory and speed using these parameters:
* **`--dit_offload_device "cpu"`**
  * **What it does:** Offloads the heavy 7B DiT model parameters to the CPU RAM, bringing them back chunk-by-chunk to stay under the 15GB VRAM limit.
* **`--vae_offload_device "cuda"`**
  * **What it does:** Keeps the VAE (encoder/decoder) on the GPU (`cuda`) for instantaneous latency improvements during initial loading and final save operations.
* **`--tensor_offload_device "cuda"`**
  * **What it does:** Forces core tensor computations to happen directly on the GPU for maximum runtime speed.
* **`--blocks_to_swap "36"`**
  * **What it does:** Determines how many model layers are swapped dynamically to preserve VRAM.

### 🎨 Extra Tweaks
* **`--vae_encode_tiled` / `--vae_decode_tiled`:** Splits large images into small grid tiles during processing to prevent VRAM usage from exploding on ultra-high-resolution images.
* **`--attention_mode "sdpa"`:** Uses PyTorch's native Scaled Dot Product Attention for speed.
* **`--seed "42"`:** Ensures deterministic, reproducible generation structures.

In [ ]:
import sys
try:
    import torch
    print("Python:", sys.version)
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
except Exception as e:
    print("Runtime check failed:", repr(e))

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
VRAM (GiB): 14.56


In [ ]:
import os, sys, subprocess

REPO = "/content/ComfyUI-SeedVR2_VideoUpscaler"
REPO_URL = "https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler"

if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
else:
    print("Repository already exists:", REPO)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", os.path.join(REPO, "requirements.txt")],
    check=True
)
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=False
)
print("Setup complete.")

Repository already exists: /content/ComfyUI-SeedVR2_VideoUpscaler
Setup complete.


In [ ]:
import os, sys, subprocess

cli_path = os.path.join(REPO, "inference_cli.py")
assert os.path.isfile(cli_path), f"CLI not found: {cli_path}"

result = subprocess.run(
    [sys.executable, cli_path, "--help"],
    cwd=REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(result.stdout[:12000])
if result.returncode != 0:
    raise RuntimeError("SeedVR2 CLI help failed.")
required = [
    "seedvr2_ema_7b_sharp-Q4_K_M.gguf",
    "--max_resolution", "--blocks_to_swap",
    "--vae_encode_tiled", "--vae_decode_tiled"
]
missing = [x for x in required if x not in result.stdout]
if missing:
    raise RuntimeError(f"Expected CLI options not found: {missing}")
print("CLI verification passed.")

/content/ComfyUI-SeedVR2_VideoUpscaler/src/common/config.py:25: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 for migration instructions.

  OmegaConf.register_new_resolver("eval", eval)
📊 Initial CUDA memory: 14.46GB free / 14.56GB total
⚠️  SeedVR2 optimizations check: SageAttention ❌ | Flash Attention ❌ | Triton ✅
💡 Optional: pip install sageattention flash-attn
🔧 Conv3d workaround active: PyTorch 2.11.0, cuDNN 92700 (fixing VAE 3x memory bug)
usage: inference_cli.py [-h] [--output OUTPUT]
                        [--output_format {mp4,png,None}]
                        [--video_backend {opencv,ffmpeg}] [--10bit]
                        [--model_dir MODEL_DIR]
                        [--dit_model {seedvr2_ema_3b-Q4_K_M.gguf,seedvr2_ema_3b-Q8_0.gguf,seedvr2_ema_3b_fp8_e4m3fn.safetensors,seedvr2_ema_3b_fp16.safetensors,seedvr2_ema_7b-Q4_K_M.gguf,see

In [ ]:
from google.colab import files
import os, zipfile, shutil

INPUT_DIR = "/content/seedvr2_input"
SUPPORTED_EXTS = (".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff")

uploaded = files.upload()
zip_candidates = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_candidates:
    raise ValueError("No .zip file uploaded. Run this cell again and choose a ZIP archive.")

ZIP_PATH = os.path.abspath(zip_candidates[0])
print("Uploaded archive:", ZIP_PATH)

if os.path.exists(INPUT_DIR):
    shutil.rmtree(INPUT_DIR)
os.makedirs(INPUT_DIR, exist_ok=True)

root_real = os.path.realpath(INPUT_DIR)
extracted_count = 0
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    for member in archive.infolist():
        if member.is_dir():
            continue
        target = os.path.realpath(os.path.join(INPUT_DIR, member.filename))
        if not target.startswith(root_real + os.sep):
            print("Skipping unsafe archive path:", member.filename)
            continue
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with archive.open(member, "r") as src, open(target, "wb") as dst:
            shutil.copyfileobj(src, dst)
        extracted_count += 1

image_paths = []
for root, _, names in os.walk(INPUT_DIR):
    for name in names:
        if name.lower().endswith(SUPPORTED_EXTS):
            image_paths.append(os.path.join(root, name))
image_paths.sort()

print(f"Files extracted: {extracted_count}")
print(f"Recognized images: {len(image_paths)}")
print("Input directory:", INPUT_DIR)
for path in image_paths[:20]:
    print(" -", os.path.relpath(path, INPUT_DIR))
if not image_paths:
    print("No supported images found. Check the ZIP contents before continuing.")

Saving gigifak.zip to gigifak.zip
Uploaded archive: /content/gigifak.zip
Files extracted: 7
Recognized images: 7
Input directory: /content/seedvr2_input
 - photo_2026-10-26_23-25-21.jpg
 - photo_2026-10-26_23-25-22.jpg
 - photo_2026-10-26_23-25-23.jpg
 - photo_2026-10-26_23-25-24.jpg
 - photo_2026-10-26_23-25-28.jpg
 - photo_2026-10-26_23-25-32.jpg
 - photo_2026-10-26_23-25-33.jpg


In [ ]:
import os, sys, subprocess
from PIL import Image

OUTPUT_DIR = "/content/seedvr2_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

image_paths = []
for root, _, names in os.walk(INPUT_DIR):
    for name in names:
        if name.lower().endswith(SUPPORTED_EXTS):
            image_paths.append(os.path.join(root, name))
image_paths.sort()
if not image_paths:
    raise RuntimeError("No input images found. Run Cell 4 and check the ZIP contents.")

successful, failed, skipped = [], [], []

for index, image_path in enumerate(image_paths, 1):
    relative_path = os.path.relpath(image_path, INPUT_DIR)
    relative_stem = os.path.splitext(relative_path)[0]
    output_path = os.path.join(OUTPUT_DIR, relative_stem + ".png")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    if os.path.isfile(output_path) and os.path.getsize(output_path) > 0:
        try:
            with Image.open(output_path) as existing:
                existing.verify()
            skipped.append(relative_path)
            print(f"[{index}/{len(image_paths)}] SKIP existing output: {relative_path}")
            continue
        except Exception:
            print("Existing output invalid; rerunning:", relative_path)

    print(f"\n[{index}/{len(image_paths)}] Processing: {relative_path}")
    try:
        with Image.open(image_path) as source:
            print("  Source dimensions:", source.size)

        # --- CONFIGURATION PARAMETERS ---
        cmd = [
            sys.executable,
            os.path.join(REPO, "inference_cli.py"),
            image_path,
            "--output", output_path,
            "--dit_model", "seedvr2_ema_7b_sharp-Q4_K_M.gguf",
            "--resolution", "2160",
            "--max_resolution", "3840",
            "--batch_size", "1",
            "--blocks_to_swap", "36",
            "--swap_io_components",
            "--dit_offload_device", "cpu",
            "--vae_offload_device", "cpu",
            "--tensor_offload_device", "cpu",
            "--vae_encode_tiled",
            "--vae_decode_tiled",
            "--attention_mode", "sdpa",
            "--seed", "42"
        ]

        # Capture stdout and stderr to debug the error
        result = subprocess.run(
            cmd,
            cwd=REPO,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        if result.returncode == 0 and os.path.isfile(output_path) and os.path.getsize(output_path) > 0:
            with Image.open(output_path) as output:
                output.verify()
            with Image.open(output_path) as output:
                print("  Output dimensions:", output.size)
            successful.append(relative_path)
            print("  SUCCESS:", output_path)
        else:
            failed.append(relative_path)
            print(f"  FAILED, exit code: {result.returncode}")
            print("\n--- CLI ERROR LOG ---")
            print(result.stdout)
            print(result.stderr)
            print("---------------------\n")
    except Exception as exc:
        failed.append(relative_path)
        print("  ERROR:", repr(exc))

print("\n========== BATCH SUMMARY ==========")
print("Total input images:", len(image_paths))
print("Successful this run:", len(successful))
print("Skipped existing outputs:", len(skipped))
print("Failed:", len(failed))
if failed:
    print("Failed files:")
    for name in failed:
        print(" -", name)


[1/7] Processing: photo_2026-10-26_23-25-21.jpg
  Source dimensions: (992, 1280)
  Output dimensions: (2160, 2786)
  SUCCESS: /content/seedvr2_output/photo_2026-10-26_23-25-21.png

[2/7] Processing: photo_2026-10-26_23-25-22.jpg
  Source dimensions: (992, 1280)
  Output dimensions: (2160, 2786)
  SUCCESS: /content/seedvr2_output/photo_2026-10-26_23-25-22.png

[3/7] Processing: photo_2026-10-26_23-25-23.jpg
  Source dimensions: (992, 1280)
  Output dimensions: (2160, 2786)
  SUCCESS: /content/seedvr2_output/photo_2026-10-26_23-25-23.png

[4/7] Processing: photo_2026-10-26_23-25-24.jpg
  Source dimensions: (992, 1280)
  Output dimensions: (2160, 2786)
  SUCCESS: /content/seedvr2_output/photo_2026-10-26_23-25-24.png

[5/7] Processing: photo_2026-10-26_23-25-28.jpg
  Source dimensions: (992, 1280)
  Output dimensions: (2160, 2786)
  SUCCESS: /content/seedvr2_output/photo_2026-10-26_23-25-28.png

[6/7] Processing: photo_2026-10-26_23-25-32.jpg
  Source dimensions: (992, 1280)
  Output dime

In [ ]:
import os, shutil
from google.colab import files

INPUT_DIR = "/content/seedvr2_input"
OUTPUT_DIR = "/content/seedvr2_output"
ZIP_OUTPUT = "/content/seedvr2_upscaled_photos.zip"

png_count = sum(
    1 for root, _, names in os.walk(OUTPUT_DIR)
    for name in names if name.lower().endswith(".png")
)
if png_count == 0:
    raise RuntimeError("No PNG outputs found. Run Cell 5 and check its batch summary.")

if os.path.exists(ZIP_OUTPUT):
    os.remove(ZIP_OUTPUT)

shutil.make_archive("/content/seedvr2_upscaled_photos", "zip", root_dir=OUTPUT_DIR)
print(f"Packaged {png_count} PNG file(s). Starting download...")
files.download(ZIP_OUTPUT)

# Clear input and output directories to ensure a clean slate for the next run
print("Cleaning up server directories...")
for folder in [INPUT_DIR, OUTPUT_DIR]:
    if os.path.exists(folder):
        for filename in os.listdir(folder):
            file_path = os.path.join(folder, filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f"Failed to delete {file_path}: {e}")

print("Cleanup complete. All input and output images have been removed from the server.")

Packaged 7 PNG file(s). Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Cleaning up server directories...
Cleanup complete. All input and output images have been removed from the server.


In [ ]:
from PIL import Image
import os

OUTPUT_DIR = "/content/seedvr2_output"
count = 0
for root, _, names in os.walk(OUTPUT_DIR):
    for name in sorted(names):
        if name.lower().endswith(".png"):
            path = os.path.join(root, name)
            try:
                with Image.open(path) as img:
                    print(f"{os.path.relpath(path, OUTPUT_DIR)}: {img.width} × {img.height}")
                count += 1
            except Exception as exc:
                print("Could not read", path, "-", repr(exc))
print(f"\nChecked {count} output image(s).")


Checked 0 output image(s).
